# Minkowski ECS — Quickstart

Minkowski is a column-oriented archetype **Entity Component System** (ECS) written in Rust.
The Python bridge lets you work with it interactively:

- **Entities** are lightweight IDs that own a set of **components** (typed data).
- **Queries** return data as Arrow tables or Polars DataFrames — one copy from BlobVec to Arrow, zero-copy from Arrow to Polars.
- **Reducers** are pre-compiled Rust closures you can call by name from Python.

## Setup

```bash
cd crates/minkowski-py
uv sync --all-extras && source .venv/bin/activate
maturin develop --release
jupyter lab notebooks/quickstart.ipynb
```

In [ ]:
import minkowski_py as mk

## Create a World

`World` is the central store for all entities and their component data.

In [ ]:
world = mk.World()
print("Entity count:", world.entity_count())
print("Components:", world.component_names())

## Spawn Entities

`spawn` creates a single entity with the given components (passed as keyword arguments).
`spawn_batch` creates many entities at once from lists of component values.

In [ ]:
import random

random.seed(42)

# Spawn a few entities one at a time (component names as one comma-separated string)
e1 = world.spawn("Position,Velocity", pos_x=10.0, pos_y=20.0, vel_x=1.0, vel_y=-1.0)
e2 = world.spawn("Position,Velocity", pos_x=50.0, pos_y=50.0, vel_x=0.5, vel_y=0.5)
print("Spawned entities:", e1, e2)

# Batch-spawn 100 entities with random positions and velocities
world.spawn_batch(
    "Position,Velocity",
    {
        "pos_x": [random.uniform(0, 500) for _ in range(100)],
        "pos_y": [random.uniform(0, 500) for _ in range(100)],
        "vel_x": [random.uniform(-5, 5) for _ in range(100)],
        "vel_y": [random.uniform(-5, 5) for _ in range(100)],
    },
)
print("Entity count after batch:", world.entity_count())

## Query as DataFrame

`world.query()` returns a **Polars DataFrame**. Internally this does one copy from
Rust BlobVec columns into Arrow arrays, then zero-copy into Polars. The result
always includes an `entity_id` column.

In [ ]:
df = world.query("Position", "Velocity")
print(df)
print("Shape:", df.shape)

## Query as Arrow

For users who prefer raw Arrow tables (e.g., for interop with DuckDB, Pandas, or other Arrow-native libraries), use `query_arrow`.

In [ ]:
table = world.query_arrow("Position", "Velocity")
print("Type:", type(table))
print(table)

## Write Back

`write_column` updates component data for existing entities. Pass the entity IDs
and the new field values as keyword arguments. Only the specified fields are overwritten.

In [ ]:
# Read current positions
df = world.query("Position", "Velocity")
entity_ids = df["entity_id"].to_list()
old_x = df["pos_x"]
old_y = df["pos_y"]

# Shift all x positions by 10
new_x = (old_x + 10.0).to_list()
new_y = old_y.to_list()

world.write_column("Position", entity_ids, pos_x=new_x, pos_y=new_y)

# Verify the update
df2 = world.query("Position")
print("First 5 rows after shifting pos_x by 10:")
print(df2.head(5))

## Reducers

Reducers are pre-compiled Rust closures registered by name. They run native Rust
logic over the ECS data — much faster than Python loops. The `ReducerRegistry`
provides access to all registered reducers.

In [ ]:
registry = mk.ReducerRegistry(world)
print("Available reducers:", registry.reducer_names())

# Run the movement reducer: applies velocity to position
registry.run("movement", world, dt=0.1, world_size=500.0)

# Verify positions changed
df3 = world.query("Position")
print("\nPositions after movement step:")
print(df3.head(5))

## Entity Lifecycle

Entities can be checked for liveness and despawned. Despawning frees the entity ID
for reuse (with a bumped generation to prevent stale references).

In [ ]:
print("Before despawn:")
print(f"  Entity {e1} alive? {world.is_alive(e1)}")
print(f"  Entity count: {world.entity_count()}")

world.despawn(e1)

print("\nAfter despawn:")
print(f"  Entity {e1} alive? {world.is_alive(e1)}")
print(f"  Entity count: {world.entity_count()}")

## Spatial Grid

`SpatialGrid` provides fast spatial neighbor queries using a uniform grid.
Build it with a world size and cell size, then `rebuild` to populate from current
entity positions. Use `query_radius` to find entities near a point.

In [ ]:
grid = mk.SpatialGrid(world_size=500.0, cell_size=25.0)
grid.rebuild(world)
print("Entities in grid:", grid.entity_count)

# Pick a position from the current data and find neighbors
df = world.query("Position")
x = df["pos_x"][0]
y = df["pos_y"][0]
neighbors = grid.query_radius(x, y, 50.0)
print(f"Neighbors within radius 50 of ({x:.1f}, {y:.1f}): {len(neighbors)} found")

## Summary

The Minkowski Python bridge exposes three main objects:

| Object | Key methods |
|---|---|
| **`World`** | `spawn`, `spawn_batch`, `query`, `query_arrow`, `write_column`, `despawn`, `is_alive`, `entity_count` |
| **`ReducerRegistry`** | `reducer_names`, `run` — call pre-compiled Rust reducers by name |
| **`SpatialGrid`** | `rebuild`, `query_radius` — uniform grid for fast neighbor lookups |

To register new component types or reducers on the Rust side, see the `minkowski-py` crate source.